[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_1/L3.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 1 | BÀI 3


---

# **MÔ HÌNH CÓ GIÁM SÁT: PHÂN LOẠI**

|  |  |
|:---|:---|
|**Thời gian đọc** |  60 phút |
|**Kiến thức nền** | Hồi quy tuyến tính, Siêu tham số  |
|**Từ khóa** | Hồi quy logistic, ma trận nhầm lẫn  |

---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q yfinance
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_1/"

*Ở hai bài trước, ta đã tìm hiểu mô hình hồi quy tuyến tính có giám sát. Trong bài này, ta chuyển sang các phương pháp phân loại.*

## **1. Giới thiệu các mô hình phân loại**

Ở Bài 1, ta đã nêu rằng các nhiệm vụ học có giám sát phổ biến nhất là hồi quy (dự báo giá trị) và phân loại (dự báo nhóm). Trong hai bài đầu, ta đã khám phá một nhiệm vụ hồi quy là dự báo lợi suất cổ phiếu, bằng hồi quy tuyến tính và các phương pháp điều chuẩn hóa. Bây giờ ta chuyển sang các phương pháp phân loại.

### **1.1 Xung lượng chuỗi thời gian: Ngưỡng**

Ta xét lại bài toán dự báo lợi suất đã học ở Bài 1. Thay vì dự báo giá trị thực của lợi suất, giờ ta chỉ quan tâm đến việc dự báo thị trường đi lên hay đi xuống. Như vậy, ta sẽ phân loại thị trường vào hai nhóm: lợi suất dương và lợi suất âm.

Tiếp theo, như ở các bài trước, ta nạp thông tin cần thiết để huấn luyện và đánh giá mô hình dự báo.

In [ ]:
import numpy as np
import yfinance as yf

# Getting historical market data from SPY (ETF) (SPY)
df = yf.download("SPY", start="2000-01-01", end="2022-01-01")

df["Ret"] = df["Adj Close"].pct_change()

name = "Ret"
df["Ret10_i"] = (
    df[name].rolling(10).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 10) - 1))
)
df["Ret25_i"] = (
    df[name].rolling(25).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 25) - 1))
)
df["Ret60_i"] = (
    df[name].rolling(60).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 60) - 1))
)
df["Ret120_i"] = (
    df[name].rolling(120).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 120) - 1))
)
df["Ret240_i"] = (
    df[name].rolling(240).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 240) - 1))
)

del df["Open"]
del df["Close"]
del df["High"]
del df["Low"]
del df["Volume"]
del df["Adj Close"]

df = df.dropna()
df.tail(10)

In [ ]:
df["Ret25"] = df["Ret25_i"].shift(-25)
df = df.dropna()
df.tail(10)

Bây giờ ta chuyển nhãn liên tục $y$ trước đó thành biến nhận giá trị $\{-1,1\}$, cho biết thị trường đã giảm hay tăng trong khoảng 25 ngày.

In [ ]:
df["Output"] = df["Ret25"].apply(np.sign)
del df["Ret25"]
df.tail(10)

In [ ]:
df.describe()

Ta chia thông tin thành tập huấn luyện và tập kiểm tra. Bên dưới là chuỗi thời gian của nhãn mà ta muốn dự báo.

In [ ]:
X, y = df.iloc[:, 0:-1], df.iloc[:, -1]
print(X.shape, y.shape)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=int(len(y) * 0.5), shuffle=False
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 7))
plt.plot(
    df.index,
    np.cumsum(y),
    label="Cumulative ups (+1) and downs (-1) of SPY over 25 days",
)
legend = plt.legend(loc="upper left")
# plt.ylim([-1.25,1.25])
plt.show()

### **1.2 Chuẩn hóa thang đo**

Một bước quan trọng trước khi huấn luyện mô hình ML là tiền xử lý dữ liệu. Ta sẽ học nhiều kỹ thuật ở module cuối, nhưng ở đây ta giới thiệu một trong những bước quan trọng nhất: *chuẩn hóa thang đo đặc trưng* (feature scaling). Các thuật toán học máy hoạt động kém khi các đặc trưng đầu vào có thang đo rất khác nhau, điều này có xu hướng làm chậm và làm xấu hiệu suất của các thuật toán tối ưu. Có hai cách phổ biến để đưa mọi thuộc tính về cùng thang đo: min-max scaling và chuẩn hóa z-score (standardization).

Min-max scaling (nhiều người gọi là chuẩn hóa chuẩn tắc) khá đơn giản: các giá trị được dịch và co giãn để nằm trong khoảng từ 0 đến 1. Ta làm điều này bằng cách trừ đi giá trị nhỏ nhất rồi chia cho hiệu giữa giá trị lớn nhất và nhỏ nhất. Scikit-Learn cung cấp bộ biến đổi `MinMaxScaler` cho việc này. Về mặt công thức, mỗi đặc trưng $j$ của quan sát $i$, $x_j^{(i)}$, được biến đổi như sau:
$$
\begin{align*}
x_j^{(i)}\leftarrow \frac{x_j^{(i)} - min_j}{max_j - min_j}
\end{align*} 
$$
trong đó $min_j$ và $max_j$ lần lượt là giá trị nhỏ nhất và lớn nhất của đặc trưng thứ $j$.

Standardization thì khác hẳn: đầu tiên trừ đi giá trị trung bình (nên các giá trị đã chuẩn hóa luôn có trung bình bằng không), sau đó chia cho độ lệch chuẩn để phân phối thu được có phương sai bằng một. Không như min-max scaling, standardization không giới hạn giá trị trong một khoảng cụ thể, điều này có thể là vấn đề với một số thuật toán. Scikit-Learn cung cấp bộ biến đổi `StandardScaler` để chuẩn hóa.

Một dạng tiền xử lý đặc trưng khác gọi là *làm trắng* (whitening): hệ trục tọa độ được xoay để tạo ra một tập đặc trưng mới không tương quan với nhau, mỗi đặc trưng được co giãn về phương sai bằng một. Thông thường, Phân tích thành phần chính (PCA, một phương pháp ML không giám sát) được dùng để đạt mục tiêu này.

Điều quan trọng là chỉ khớp các bộ chuẩn hóa trên dữ liệu huấn luyện, không phải trên toàn bộ tập dữ liệu (gồm cả tập kiểm tra). Chỉ khi đó ta mới dùng chúng để biến đổi tập huấn luyện và tập kiểm tra (và dữ liệu mới).

Đoạn mã dưới đây chuẩn hóa các quan sát trong dữ liệu của ta.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_input = MinMaxScaler(feature_range=(-1, 1))
scaler_input.fit(X_train)
X_train = scaler_input.transform(X_train)
X_test = scaler_input.transform(X_test)

## **2. Mô hình hồi quy logistic**

Hồi quy logistic (còn gọi là hồi quy Logit) thường được dùng để ước lượng xác suất một quan sát thuộc về một nhóm cụ thể (ví dụ: xác suất thị trường đi lên là bao nhiêu?). Nếu xác suất ước lượng lớn hơn 50%, mô hình dự báo quan sát đó thuộc nhóm ấy (gọi là nhóm dương, gắn nhãn "1"), ngược lại thì dự báo nó không thuộc (tức thuộc nhóm âm, gắn nhãn "0"). Điều này khiến nó trở thành một bộ phân loại nhị phân.

Giả sử ta có mẫu huấn luyện với các quan sát mà nhãn $y$ chỉ đơn giản cho biết thị trường đã tăng $y=1$ hay giảm $y=0$. Giống như mô hình hồi quy tuyến tính, mô hình hồi quy logistic tính tổng có trọng số của các đặc trưng đầu vào (cộng số hạng chệch), nhưng thay vì xuất kết quả trực tiếp như hồi quy tuyến tính, nó xuất hàm logistic của kết quả đó:
$$
\begin{align*}
\widehat{p}^{(i)}=\sigma\left(x^{(i)}\theta\right)
\end{align*}
$$
trong đó $\sigma(.)$ là hàm sigmoid được định nghĩa là:
$$
\begin{align*}
\sigma(z)=\frac{1}{1+\exp(-z)}
\end{align*}
$$
Khi đó, dự báo của mô hình hồi quy logistic là:
$$
\begin{align*}
\widehat{y}^{(i)}=\begin{cases}
                   0\ \text{if}\ \widehat{p}^{(i)}\lt 0.5 \\
                                1\ \text{if}\ \widehat{p}^{(i)}\geq 0.5 
                  \end{cases}
\end{align*}
$$
Lưu ý rằng hồi quy logistic dự báo 1 nếu $x^{(i)}\theta$ dương và 0 trong trường hợp còn lại.

### **2.1 Huấn luyện và hàm chi phí**

Mục tiêu của huấn luyện là chọn vector tham số $\theta$ sao cho mô hình ước lượng xác suất cao cho các quan sát dương ($y^{(i)}$ = 1) và xác suất thấp cho các quan sát âm ($y^{(i)}$ = 0). Ý tưởng này được thể hiện bằng hàm chi phí sau:
$$
\begin{align*}
J(\theta) = - \frac{1}{m}\sum_{i=1}^m \left[y^{(i)}log\left(\widehat{p}^{(i)}\right) + \left(1-y^{(i)}\right)log\left(1 - \widehat{p}^{(i)}\right) \right]
\end{align*}
$$
Hàm chi phí này hợp lý vì $-log\left(\widehat{p}^{(i)}\right)$ rất lớn khi $\widehat{p}^{(i)}$ tiến về 0, nên chi phí sẽ lớn nếu mô hình ước lượng xác suất gần 0 cho một quan sát có $y^{(i)}$ = 1; tương tự với $y^{(i)}$ = 0 khi $\widehat{p}^{(i)}$ tiến về 1. Thật vậy, trong Kinh tế lượng, hàm trên chính là hàm log-hợp lý của mô hình Logit.

Mặc dù không có nghiệm dạng đóng cho vector $\theta$ cực tiểu hóa hàm chi phí trên, ta biết hàm này là hàm lồi, nên chẳng hạn Gradient Descent được đảm bảo tìm ra cực tiểu toàn cục miễn là tốc độ học đủ nhỏ và ta chờ đủ lâu. Do đó, nếu dùng thuật toán Gradient Descent (Batch, Mini-batch hoặc Stochastic), ta dùng các đạo hàm riêng sau của hàm chi phí:
$$
\begin{align*}
\frac{\partial}{\partial\theta_j}J(\theta) = -\frac{1}{m}\sum_{i=1}^m \left[y^{(i)} - \sigma\left(x^{(i)}\theta\right) \right]x_j^{(i)}
\end{align*}
$$
Giống như hồi quy tuyến tính, hồi quy logistic có thể được điều chuẩn hóa bằng các phạt Ridge và Lasso đã mô tả ở Bài 2. Thực tế Scikit-Learn mặc định thêm phạt $\ell_2$ (Ridge).

Ta huấn luyện mô hình dự báo thị trường chứng khoán bằng thông tin chuỗi thời gian với hồi quy logistic.

*Lưu ý: nếu muốn thử thay đổi cường độ điều chuẩn hóa, ta cần đưa tham số đầu vào "C" vào lớp `logisticRegr` được định nghĩa bên dưới, trong đó "C" là nghịch đảo của cường độ điều chuẩn hóa.*

In [ ]:
from sklearn.linear_model import LogisticRegression

# all parameters not specified are set to their defaults
logisticRegr = LogisticRegression()
logisticRegr.fit(X_train, y_train)

In [ ]:
predictions = logisticRegr.predict(X_test)

In [ ]:
# Use score method to get accuracy of model
score = logisticRegr.score(X_test, y_test)
print(score)

Mô hình dự báo đúng kết quả thực tế 72% số lần. Con số này có vẻ khá đáng kể với một mô hình đơn giản như vậy, đúng không? Bên dưới ta sẽ chỉ ra rằng con số này có thể gây hiểu lầm khá nhiều.

Trước khi mô tả chi tiết các thước đo hiệu suất của bộ phân loại, ta mô tả trước một cách khái quát hóa đơn giản của hồi quy logistic cho trường hợp nhiều nhóm.

### **2.2 Hồi quy Softmax**

Mô hình hồi quy logistic có thể khái quát hóa cho nhiều nhóm mà không cần huấn luyện rồi kết hợp nhiều bộ phân loại nhị phân. Đây gọi là *Hồi quy Softmax* hay *Hồi quy Logit đa thức*. Ví dụ, giả sử ta muốn thêm một nhóm mới vào mô hình dự báo lợi suất để thể hiện các biến động nhỏ của lợi suất thị trường, chẳng hạn trong khoảng [-0,5%, +0,5%]. Khi đó ta có thể huấn luyện một mô hình dự báo xác suất lợi suất thị trường là "cao" (>0,5%), thấp (<-0,5%) hoặc "trung gian".

Mô hình hồi quy Softmax trước hết tính điểm cho từng quan sát $i$ và từng nhóm $c$, $x^{(i)}\theta^{(c)}$, rồi ước lượng xác suất của mỗi nhóm bằng hàm softmax. Lưu ý rằng mỗi nhóm có một vector tham số riêng, $\theta^{(c)}$, có số chiều bằng số đặc trưng đầu vào cộng một số hạng chệch.

Khi đó, nếu có $C$ nhóm khác nhau, xác suất ước lượng của nhóm $c$ đối với quan sát $i$ bằng hàm mũ của điểm số của nó, chuẩn hóa bằng tổng hàm mũ của điểm số của tất cả các nhóm.
$$
\begin{align*}
\widehat{p}_c^{(i)}=Softmax_c\left(x^{(i)},\Theta\right)=\frac{exp\left(x^{(i)}\theta^{(c)}\right)}{\sum_{j=1}^C exp\left(x^{(i)}\theta^{(j)}\right)}
\end{align*}
$$
trong đó $\Theta$ gom tất cả các tham số của mô hình. Dự báo của mô hình $\widehat{y}^{(i)}$ là nhóm có điểm số $x^{(i)}\theta^{(c)}$ cao nhất.

Tương tự hồi quy logistic, hàm chi phí để huấn luyện các tham số của mô hình có thể định nghĩa là:
$$
\begin{align*}
J(\Theta) = - \frac{1}{m}\sum_{i=1}^m\sum_{c=1}^C y^{(i)}log\left(\widehat{p}_c^{(i)}\right) 
\end{align*}
$$
thường gọi là hàm chi phí *entropy chéo* (cross-entropy). Các phần tử tương ứng của vector gradient là:
$$
\begin{align*}
\frac{\partial}{\partial\theta_k^{(c)}}J(\Theta) = -\frac{1}{m}\sum_{i=1}^m \left[y^{(i)} - Softmax_c\left(x^{(i)},\Theta\right) \right]x_k^{(i)}
\end{align*}
$$
Từ đó ta có thể dùng tối ưu Gradient Descent.

## **3. Các thước đo hiệu suất trong phân loại**

Đánh giá một bộ phân loại thường khó hơn đáng kể so với đánh giá một mô hình hồi quy. Một thước đo hiệu suất đơn giản có thể là tần suất bộ phân loại phân loại đúng các nhãn trong mẫu kiểm tra, tức *độ chính xác* (accuracy). Tuy nhiên chỉ số này có thể gây hiểu lầm. Trong mô hình dự báo thị trường chứng khoán, ta thấy tỷ lệ thành công 72% khi dự báo thị trường tăng hay giảm. Nhưng sức dự báo đó có thể chỉ đơn giản đến từ việc mô hình phần lớn thời gian dự báo thị trường tăng, điều xảy ra với xác suất khoảng 72%. Thực tế đúng là như vậy, như ta sẽ chỉ ra bên dưới.

Tương tự, mô hình phân loại có thể nhắm đến việc dự báo các nhóm hoặc sự kiện tương đối hiếm, chẳng hạn một sự kiện hay nhóm có xác suất 5%. Trong những trường hợp đó, ngay cả một bộ phân loại rất tệ cũng có thể đạt độ chính xác cao. Ví dụ, nếu luôn dự báo rằng sự kiện không bao giờ xảy ra, ta đương nhiên đạt độ chính xác 95%. Để khắc phục điều này và chọn mô hình phân loại tốt hơn, có nhiều thước đo hiệu suất như ta nêu dưới đây.

### **3.1 Ma trận nhầm lẫn**

Một cách tốt hơn nhiều để đánh giá hiệu suất của bộ phân loại là xem ma trận nhầm lẫn (confusion matrix). Ý tưởng chung là đếm số lần các quan sát của nhóm A bị phân loại thành nhóm B. Để tính ma trận nhầm lẫn, trước hết ta cần có một tập dự báo để so sánh với các mục tiêu thực tế. Mỗi hàng trong ma trận nhầm lẫn là một nhóm thực tế, còn mỗi cột là một nhóm dự báo. Một bộ phân loại hoàn hảo chỉ có dương tính thật và âm tính thật, nên ma trận nhầm lẫn của nó chỉ có giá trị khác không trên đường chéo chính (từ trên trái xuống dưới phải).

Với bộ phân loại hai nhóm, trong đó 0 là nhóm âm và 1 là nhóm dương, ta biểu diễn ma trận nhầm lẫn như sau:
$$
\begin{align*}
Confusion = \pmatrix{True\ Negatives & False\ Positives \\
                      False\ Negatives & True\ Positives }
\end{align*}
$$
(True Negatives: âm tính thật; False Positives: dương tính giả; False Negatives: âm tính giả; True Positives: dương tính thật.)

Ta hãy quan sát ma trận nhầm lẫn của bộ phân loại.

In [ ]:
# import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics

cm = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(9, 9))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=0.5, square=True, cmap="Blues_r")
plt.ylabel("Actual label")
plt.xlabel("Predicted label")
all_sample_title = "Accuracy Score: {0}".format(score)
plt.title(all_sample_title, size=15);

Thật vậy, tỷ lệ thành công 72% của mô hình đến từ việc nó dự báo thị trường chứng khoán đi lên, điều xảy ra với xác suất gần bằng tỷ lệ thành công đó. Do đó, ta có thể kết luận mô hình của mình là một bộ phân loại kém để dự báo lợi suất thị trường chứng khoán.

## **3.2 Precision và Recall**
Mức độ chính xác của các dự báo dương có thể đo bằng precision (độ chính xác dương) của bộ phân loại:
$$
\begin{align*}
Precision = \frac{TP}{TP + FP}
\end{align*}
$$
trong đó $TP$ là tổng số dương tính thật và $FP$ là số dương tính giả. Nghĩa là mẫu số chứa số lần bộ phân loại đã dự báo 1, nên precision cho biết tỷ lệ dự báo đúng trên tất cả các giá trị 1 mà mô hình đã dự báo.

Precision thường được kết hợp với một thước đo khác là recall (độ bao phủ):
$$
\begin{align*}
Recall = \frac{TP}{TP + FN}
\end{align*}
$$
Recall cho biết tỷ lệ dự báo đúng trên tất cả các giá trị 1 thực sự có trong mẫu.

Precision và recall thường được gộp thành một thước đo duy nhất gọi là điểm $F_1$, là trung bình điều hòa của hai giá trị, trong đó giá trị thấp có trọng lượng lớn hơn nhiều. Do đó, điểm $F_1$ chỉ cao khi cả precision và recall cùng cao:
$$
\begin{align*}
F_1 = 2\frac{Precision \times Recall}{Precision + Recall}
\end{align*}
$$

Điểm $F_1$ ưu ái các bộ phân loại có precision và recall xấp xỉ nhau. Đôi khi ta không muốn như vậy. Chẳng hạn, nếu xây dựng mô hình phát hiện xác suất thị trường chứng khoán sụt giảm mạnh (khủng hoảng), khi đó nhóm cần dự báo là 1, ta có thể sẵn sàng hy sinh một phần precision (dự báo có khủng hoảng trong khi không xảy ra) để đổi lấy recall cao hơn (không bỏ sót khủng hoảng khi nó thực sự xảy ra).

Đáng tiếc là ta không thể có bộ phân loại vừa có precision cao vừa có recall cao cùng lúc. Để thấy điều này, lưu ý rằng các mô hình như hồi quy logistic trả về xác suất chứ không phải kết quả rời rạc. Để xác định mô hình đang dự báo nhóm nào, ta đặt một giá trị ngưỡng trên xác suất dự báo để phân biệt nhóm dương và nhóm âm, như ngưỡng 0,5 trong hồi quy logistic ở trên. Tùy giá trị ngưỡng, nhóm dự báo của một số quan sát có thể thay đổi. Nếu bộ phân loại đặt ngưỡng cao để tăng precision, số trường hợp dương bị dự báo thành âm sẽ tăng, làm giảm recall. Ngược lại, nếu đặt ngưỡng cao để tăng recall thì phải trả giá bằng việc dự báo nhiều trường hợp âm thành dương hơn, làm giảm precision.

### **3.3 ROC và AUC**

Đường cong ROC (Receiver Operating Characteristic) là một công cụ phổ biến khác dùng với bộ phân loại nhị phân. ROC vẽ recall của bộ phân loại theo *tỷ lệ dương tính giả* (false positive rate), tức tỷ lệ các giá trị 0 bị phân loại thành 1. Giống như sự đánh đổi giữa precision và recall, khi bộ phân loại tăng recall thì nó sẽ tạo ra nhiều dương tính giả hơn. Một cách so sánh các bộ phân loại là dùng Diện tích dưới đường cong (AUC), cho ta diện tích nằm dưới ROC của bộ phân loại. Với bộ phân loại hoàn hảo, AUC bằng một. Với bộ phân loại hoàn toàn ngẫu nhiên, AUC bằng 0,5.

Nên dùng thước đo nào để xác định bộ phân loại tốt? Ta nên dùng $F_1$ khi quan tâm nhiều hơn đến các dương tính giả hoặc khi nhóm dương hiếm.

Đoạn mã dưới đây vẽ đường cong ROC cho mô hình hồi quy logistic của ta. Để có được nó, hàm roc_curve() tính recall (hay True Positive Rate) và False Positive Rate trên mẫu kiểm tra cho nhiều giá trị ngưỡng quyết định dự báo của mô hình. Hãy chú ý khả năng dự báo của mô hình thuộc loại *kém nhất trong những mô hình kém*.

In [ ]:
from sklearn import metrics

fpr, tpr, thresholds = metrics.roc_curve(y_test, predictions)
roc_auc = metrics.auc(fpr, tpr)
display = metrics.RocCurveDisplay(
    fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name="estimation"
)
display.plot()
plt.show()

## **4. Kết luận**

Đến đây, ta kết thúc phần mô tả các bài toán phân loại cơ bản trong học máy trong khuôn khổ hồi quy logistic. Ở Bài 4, ta sẽ xây dựng một ứng dụng chấm điểm tín dụng.

Hẹn gặp lại bạn ở bài sau!